In [70]:
import os
import logging
from typing import List, Dict, Tuple
from collections import Counter

import freecad
import FreeCAD
import Import
import Part
import os, shutil, json, time
from pathlib import Path
import pandas as pd


In [30]:
# Konfiguration
project_path = Path(os.getcwd()).parent
path_to_Gears = project_path / 'data/3_primary/fabwave/Gears'
path_to_HeadlessScrews = project_path / 'data/3_primary/fabwave/HeadlessScrews'

In [31]:
gear_files = list(path_to_Gears.rglob("*.step"))
hs_files = list(path_to_HeadlessScrews.rglob("*.step"))

In [32]:
def get_volumes(step_files):
    volumes ={}
    for step_file in step_files:
        doc = FreeCAD.newDocument(f"{step_file}")
        try:
            Part.insert(step_file.as_posix(), doc.Name)
            shapes = [volume.Shape for volume in doc.Objects]

            
            fused_shape = shapes[0] 

            volumes[step_file] = fused_shape.Volume

        except Exception as e:
            print(f"ERROR: {step_file} -> {e}\n")

        FreeCAD.closeDocument(doc.Name)

    return volumes

In [33]:
gear_volumes = get_volumes(gear_files)
hs_volumes = get_volumes(hs_files)

## check if the values are unique

In [37]:
len(set(gear_volumes.values()))==len(gear_volumes.keys()), len(set(hs_volumes.values()))==len(hs_volumes.keys())

(True, True)

## check for numerical effects.
These could lead to identical parts which could not be observed by the above method

In [94]:
df_gears = pd.DataFrame.from_dict(gear_volumes, orient='index', columns=['volume']).sort_values('volume')
df_hs = pd.DataFrame.from_dict(hs_volumes, orient='index', columns=['volume']).sort_values('volume')

In [95]:
sorted_gear_volumes = list(gear_volumes.values())
sorted_gear_volumes.sort()

sorted_hs_volumes = list(hs_volumes.values())
sorted_hs_volumes.sort()

In [96]:
def get_rel_deviations(sorted_lst):
    rel_deviations = []
    # calc relative dev to the next part each
    for i in range(1,len(sorted_lst)):
        lower = sorted_lst[i-1]
        higher = sorted_lst[i]

        rel_deviation = (higher-lower)/higher

        rel_deviations.append(rel_deviation)
    
    return rel_deviations

In [97]:
gear_dev = get_rel_deviations(sorted_gear_volumes)
hs_dev = get_rel_deviations(sorted_hs_volumes)


In [98]:
df_gears['dev_to_previous']=[1]+gear_dev
df_hs['dev_to_previous']=[1]+hs_dev

In [112]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(df_gears) #.sort_values('dev_to_previous')

,volume,dev_to_previous
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/f612b549-b8d8-4049-9a9c-ec865cbd6038.step,123.809785,1.000000
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/daa35be6-2d63-4fb2-8807-fe29d5d65271.step,136.584158,0.093527
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/d58620cc-e091-4f0d-8b11-81de14ea1e00.step,152.162835,0.102382
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/fcee0a82-2afe-45fb-bc27-66bbb9657395.step,202.609786,0.248986
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/00154245-00d1-4ead-9c17-98f80254af1b.step,230.696994,0.121749
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/c31b9db4-8875-4062-bccc-036a5cee2509.step,245.811739,0.061489
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/c5b42dac-bc35-4185-8a32-2b0e608e99f9.step,415.235211,0.408018
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/e6cfa40b-d249-4edd-8c60-c66b59c9666d.step,427.138567,0.027868
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/2f09a416-430f-4786-b7ba-158dd80e00b4.step,500.820588,0.147123
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/Gears/0d2ec54a-9297-406e-a0d4-e3af3ba5f3ac.step,575.174801,0.129272


No duplicates detected in gears

In [111]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(df_hs)


,volume,dev_to_previous
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/1a360df4-13aa-4ace-a598-31b7e4d13048.step,1.379987e-01,1.000000e+00
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/e066d85f-c02f-4fd1-bb17-c0005a0e7c52.step,2.019519e-01,3.166751e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/88a4545e-5a2f-4b98-83fa-b818e270f04c.step,2.671271e-01,2.439858e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/637c68ea-6f0b-4b44-859f-0ee39f8024ab.step,3.530836e-01,2.434453e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/0aa83360-0704-4b06-8b3b-2b7774b7de92.step,4.285772e-01,1.761493e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/83648588-8d19-4013-8913-bbf76f5f7c99.step,5.368671e-01,2.017070e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/0554c639-fb1f-456f-8786-5225853908ab.step,5.994310e-01,1.043722e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/257e04d0-c8f6-4425-b66c-ef247a992e36.step,8.861768e-01,3.235763e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/3a5e90e5-4a41-415b-87c2-73adc5895a46.step,1.041999e+00,1.495416e-01
/Users/mkruse/Documents/repos/clear-shape/data/3_primary/fabwave/HeadlessScrews/2504c4b5-860f-40ef-95c7-d719ebefd065.step,1.430131e+00,2.713962e-01


In [117]:
duplicates_hs = df_hs[df_hs.dev_to_previous < 0.001]

detect 6 files which should be handled as duplicates

In [ ]:
with open(r"../reports/raw->primary/detected_duplicates.csv", 'w') as f:
    for file in list(duplicates_hs.index):
        f.write(f"{str(file)},duplicate\n")
